# 💻 Chapter 20: Hashing, Bloom Filters & Probabilistic Data Structures
*Track 2 — Developers | Tier 2*

> **🎬 Hook:** Google checks if a URL is malicious in microseconds using near-zero memory. How? They accept a small probability of being wrong — and it's a great trade.

**🎯 Objectives:** Understand and implement Bloom filters, HyperLogLog, Count-Min Sketch — trade accuracy for speed and memory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# ── Bloom Filter: Theory ──
def bloom_false_positive_prob(n, m, k):
    # P(false positive) for n items, m bits, k hash functions.
    return (1 - np.exp(-k*n/m))**k

def optimal_k(n, m):
    # Optimal number of hash functions.
    return int(np.ceil((m/n) * np.log(2)))

print("📊 Bloom Filter Design Guide")
print(f"{'Items (n)':>12} {'Bits (m)':>10} {'Bits/item':>10} {'k':>6} {'FP rate':>10}")
print("-" * 52)
for n, m in [(1000, 10000), (10000, 100000), (1000000, 10000000)]:
    k = optimal_k(n, m)
    fp = bloom_false_positive_prob(n, m, k)
    print(f"{n:>12,} {m:>10,} {m/n:>10.1f} {k:>6} {fp:>10.4%}")

# Effect of k on FP rate for fixed n and m
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
n, m = 1000, 10000
k_range = np.arange(1, 20)
fp_rates = [bloom_false_positive_prob(n, m, k) for k in k_range]
axes[0].plot(k_range, fp_rates, 'bo-', markersize=6, lw=2)
axes[0].axvline(optimal_k(n, m), color='red', lw=2, linestyle='--',
                label=f'Optimal k={optimal_k(n,m)}')
axes[0].set_xlabel('Number of hash functions (k)')
axes[0].set_ylabel('False positive probability')
axes[0].set_title(f'Bloom Filter: n={n}, m={m}', fontweight='bold')
axes[0].legend()

# Effect of bits per item
bits_per_item = np.linspace(2, 20, 100)
k_opt = np.ceil(bits_per_item * np.log(2))
fp_opt = bloom_false_positive_prob(1, bits_per_item, k_opt)
axes[1].semilogy(bits_per_item, fp_opt, 'g-', lw=3)
axes[1].set_xlabel('Bits per item (m/n)')
axes[1].set_ylabel('Optimal FP rate')
axes[1].set_title('Bloom Filter: FP rate vs Memory', fontweight='bold')
axes[1].axvline(10, color='red', lw=2, linestyle='--', label='10 bits/item → ~0.8% FP')
axes[1].legend()

plt.tight_layout()
plt.savefig('ch20_bloom_theory.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Full Bloom Filter Implementation ──
class BloomFilter:
    def __init__(self, expected_n, fp_rate=0.01):
        self.m = int(-expected_n * np.log(fp_rate) / (np.log(2)**2))
        self.k = int(np.ceil(self.m / expected_n * np.log(2)))
        self.bits = bytearray(self.m)
        self.n_added = 0

    def _hash_positions(self, item):
        item_bytes = str(item).encode()
        positions = set()
        for seed in range(self.k):
            h = hashlib.sha256(item_bytes + seed.to_bytes(4, 'big')).hexdigest()
            positions.add(int(h, 16) % self.m)
        return positions

    def add(self, item):
        for pos in self._hash_positions(item):
            self.bits[pos] = 1
        self.n_added += 1

    def __contains__(self, item):
        return all(self.bits[pos] for pos in self._hash_positions(item))

    def estimated_fp_rate(self):
        return bloom_false_positive_prob(self.n_added, self.m, self.k)

# Build URL blacklist Bloom filter
bf = BloomFilter(expected_n=10000, fp_rate=0.001)
print(f"Bloom Filter: m={bf.m:,} bits ({bf.m/8/1024:.1f} KB), k={bf.k}")
print(f"Exact set of 10,000 URLs would need ~80KB minimum")

# Add known malicious URLs
malicious = [f"malicious{i}.evil.com" for i in range(10000)]
for url in malicious:
    bf.add(url)

# Test
legit = [f"legit{i}.good.com" for i in range(10000)]
fp_count = sum(url in bf for url in legit)
print(f"
False positives on 10,000 legit URLs: {fp_count} ({fp_count/100:.1f}%)")
print(f"Theoretical FP rate: {bf.estimated_fp_rate():.3%}")
print(f"True positives: {sum(url in bf for url in malicious[:100])}/100")

In [ ]:
# ── Count-Min Sketch: Frequency Estimation ──
class CountMinSketch:
    def __init__(self, width=1000, depth=5):
        self.width = width
        self.depth = depth
        self.table = np.zeros((depth, width), dtype=np.int64)
        self.seeds = rng.integers(0, 10**9, depth)

    def _hash(self, item, i):
        h = hashlib.md5(f"{self.seeds[i]}{item}".encode()).hexdigest()
        return int(h, 16) % self.width

    def add(self, item, count=1):
        for i in range(self.depth):
            self.table[i, self._hash(item, i)] += count

    def estimate(self, item):
        return min(self.table[i, self._hash(item, i)] for i in range(self.depth))

# Word frequency estimation
words = np.random.choice(['apple','banana','cherry','date','elderberry',
                          'fig','grape','mango'] + [f'w{i}' for i in range(100)],
                         p=[0.15,0.12,0.10,0.08,0.07,0.06,0.05,0.05]+
                           [0.32/100]*100,
                         size=100000)

# True counts
true_counts = {w: (words == w).sum() for w in set(words)}

cms = CountMinSketch(width=500, depth=5)
for w in words:
    cms.add(w)

print("Count-Min Sketch vs True Counts (top words):")
print(f"{'Word':<12} {'True':>8} {'CMS Est':>8} {'Error':>8}")
top_words = sorted(true_counts.items(), key=lambda x: -x[1])[:8]
errors = []
for word, true_count in top_words:
    estimated = cms.estimate(word)
    error = (estimated - true_count) / true_count
    errors.append(abs(error))
    print(f"  {word:<12} {true_count:>8} {estimated:>8} {error:>7.2%}")
print(f"Average relative error: {np.mean(errors):.2%}")

## ✏️ Section 6 — Coding Challenges

**Challenge 1:** Implement HyperLogLog for cardinality estimation.
(Estimate the number of unique items in a stream with O(log log n) memory)

**Challenge 2:** Design a Bloom filter for a URL shortener (billions of URLs).
What are appropriate m and k values?

**Challenge 3:** Evaluate the space-accuracy tradeoff: compare exact dict vs Bloom filter vs Count-Min Sketch for 1M items.

<details><summary>Solutions</summary>
HyperLogLog: hash items, find longest leading-zero run, use harmonic mean of estimates.
URL shortener with 1B URLs and 1% FP rate: ~10 bits/item = 1.25 GB vs 8 GB for exact set.
</details>

In [ ]:
# Space comparison: exact vs probabilistic
n = 100_000
# Exact: Python set
exact = set(str(i) for i in range(n))
exact_bytes = sum(len(s) for s in exact) + n * 50  # ~50 bytes overhead per str

# Bloom filter
bf2 = BloomFilter(n, fp_rate=0.01)
for i in range(n): bf2.add(i)
bf_bytes = bf2.m // 8

# Count-Min Sketch
cms2 = CountMinSketch(width=10000, depth=5)
cms_bytes = cms2.table.nbytes

print("📊 Space Comparison for 100,000 items:")
print(f"  Exact set:         ~{exact_bytes/1024:.0f} KB (100% accurate)")
print(f"  Bloom filter:       {bf_bytes/1024:.0f} KB ({bf2.estimated_fp_rate():.2%} FP)")
print(f"  Count-Min Sketch:   {cms_bytes/1024:.0f} KB (frequency estimates)")
print(f"
Bloom filter uses {exact_bytes/bf_bytes:.0f}x less space than exact set!")

## 🎯 Recap
1. Bloom filters: space-efficient membership testing with tunable FP rate. Never false negatives.
2. Count-Min Sketch: frequency estimation with bounded error.
3. These structures power Google SafeBrowsing, Redis, Cassandra, and most distributed databases.

**Next:** [Chapter 21 — Statistical Testing for Feature Flags]